In [1]:
# This is the python script performing the analysis of many LC-MS files for RiPP modifications
import sys
import os
import pickle
import argparse

# science imports
import numpy as np
import pandas as pd
import scipy as sp
import matplotlib.pyplot as plt
import seaborn as sns
from pyteomics import mzml, mass
from pyteomics import parser as peptide_parser
from classes_spectrometry import *
from modification_dictionary import *
from pipeline_functions import *


def group_cuts_analysis(experiment: MSExperiment, path_output_folder=None):
    # we load the associated spectrum
    spectrum = MassSpectrum(mzml_path=experiment.mzml_path, output_path=experiment.output_path,
                            spectrum_id=experiment.spectrum_id)
    spectrum.load(override=False, minimum_intensity=1000)
    # we initialize the AnalysisResults object
    results = AnalysisResults(experiment)
    # we save the results object to the output folder if path_output_folder is provided

    grouped_cuts = []
    summary = []  # Initialize summary to an empty list
    for target in experiment.target_peptides:
        peak_list = find_ms_peaks_targeted_ms1(spectrum, experiment, target, mz_boundaries=2, time_boundaries=0.2,
                                               filtered=False)

        grouped_cuts.append((target, peak_list))
        results.add_peaks(target, peak_list)
        good_peaks = results.get_selected_peaks()
        summary = results.summarize()
        # we call the save method of the AnalysisResults
    results.save(override=True)
    return grouped_cuts, results, summary

In [2]:


# we import the input excel file from \input_excel_files\96_well_plate_example.xlsx
input_excel_file = r'C:\Users\jordant\PycharmProjects\LCMS_RiPP\input_excel_files\96_well_plate_expressions.xlsx'
df_samples = pd.read_excel(input_excel_file)

#we get the first experiment (point mutants)
# we filter the dataframe to get only the rows with experiment_id == 'point_mutants'
#df_samples = df_samples[df_samples['experiment_id'] == 'point_mutants']

# we split the sample_id to get plate_x and well_y
df_samples['well'] = df_samples['sample_id'].str.split('_').str[2]
# we groupby the experiment_id and sample_id
#subset = df_samples[df_samples["peptide_sequence"] == "GGCGPDPVYGVILPWA"]
grouped = df_samples.groupby(['experiment_id', 'precursor_plasmid', 'maturase_name'])

In [ ]:

for i, ((experiment_id, precursor_plasmid, maturase_name), group) in enumerate(grouped):
    try:


        report_path = os.path.join(
        fr"C:\Users\jordant\PycharmProjects\LCMS_RiPP\output_report_files_plates\{experiment_id}",
        f"{experiment_id}_{precursor_plasmid}_{maturase_name}_reports")
        # we loop over the group and create an MSExperiment object BY LOADING THE pickle file for each row
        experiments = []
        grouped_cuts_list = []
        results_list = []
        summary_list = []
        for index, row in group.iterrows():
            pickle_name = f"{row['sample_id']}_{row['experiment_id']}_experiment.pkl"
            path = os.path.join(row['output_folder_name'], pickle_name)
            with open(path, 'rb') as f:
                experiment = pickle.load(f)
                experiments.append(experiment)
            # we call the group_cuts_analysis function to analyze the experiments
            grouped_cuts, results, summary = group_cuts_analysis(experiment, report_path)
            # we save the s
            grouped_cuts_list.append(grouped_cuts)
            results_list.append(results)
            summary_list.append(summary)
        # we create a GroupedExperiments object to hold the results for this grouped_summaries we pass each result from results_list as an argument to the GroupedExperiments class
        grouped_experiment = GroupedExperiments(*results_list)
        # report name

        # we call the report function to create a report for this grouped_experiment
        grouped_experiment.generate_all_reports(output_dir=report_path)
    except:
        continue

    print("*" * 30,
          f"\nFinished analysis for group {i + 1}/{len(grouped)}: {experiment_id}, {precursor_plasmid}, {maturase_name}\n",
          "*" * 30)

Spectral data for 'plate_1_G11_point_mutants' successfully loaded from cache.
Processing charge 1


C:\Users\jordant\AppData\Local\anaconda3\envs\lcms\Lib\site-packages\scipy\signal\_peak_finding.py:1193: RuntimeWarning: divide by zero encountered in scalar divide
  snr = abs(cwt[line[0][0], line[1][0]] / noises[line[1][0]])
C:\Users\jordant\AppData\Local\anaconda3\envs\lcms\Lib\site-packages\pandas\core\indexes\base.py:945: RuntimeWarning: overflow encountered in exp
  result = getattr(ufunc, method)(*new_inputs, **kwargs)
C:\Users\jordant\AppData\Local\anaconda3\envs\lcms\Lib\site-packages\scipy\signal\_peak_finding.py:1193: RuntimeWarning: divide by zero encountered in scalar divide
  snr = abs(cwt[line[0][0], line[1][0]] / noises[line[1][0]])


Integration failed: Zoomed extracted data is missing.
Processing charge 2
Parameters: (np.float64(581.1854069663775), np.float64(5.674770600000032), 0.03, 0.1, 21500.3017578125)


C:\Users\jordant\AppData\Local\anaconda3\envs\lcms\Lib\site-packages\scipy\signal\_peak_finding.py:1193: RuntimeWarning: invalid value encountered in scalar divide
  snr = abs(cwt[line[0][0], line[1][0]] / noises[line[1][0]])
C:\Users\jordant\AppData\Local\anaconda3\envs\lcms\Lib\site-packages\scipy\signal\_peak_finding.py:1193: RuntimeWarning: divide by zero encountered in scalar divide
  snr = abs(cwt[line[0][0], line[1][0]] / noises[line[1][0]])
C:\Users\jordant\PycharmProjects\LCMS_RiPP\classes_spectrometry.py:714: OptimizeWarning: Covariance of the parameters could not be estimated
  gaussian_params, pcov = sp.optimize.curve_fit(


Processing charge 3
Skipping charge 3 due to insufficient data points
Processing charge 1


In [ ]:

# we load the "C:\Users\jordant\PycharmProjects\LCMS_RiPP\output_data_plates\point_mutants_plate_1_G5\plate_1_G5_point_mutants_results.pkl"
with open(
        r"C:\Users\jordant\PycharmProjects\LCMS_RiPP\output_report_files_plates\point_mutants\point_mutants_TJ-C-509_HypX-HypY_reports\plotting_data.pkl",
        'rb') as f:
    loaded_results = pickle.load(f)
# we can access the results


with open(
        r"C:\Users\jordant\PycharmProjects\LCMS_RiPP\output_data_plates\point_mutants_plate_3_G11\plate_3_G11_point_mutants_results.pkl",
        "rb") as f:
    plate_3_G11_results = pickle.load(f)
for peaks in plate_3_G11_results.selected_peaks.values():
    for peak in peaks:
        peak.plot_peak()

peaks = plate_3_G11_results.found_peaks.values()
peaks = [peak for sublist in peaks for peak in sublist]  # Flatten the list of lists
# we sort by inverse trapz area


with open(
        r"C:\Users\jordant\PycharmProjects\LCMS_RiPP\output_data_plates\point_mutants_plate_3_H11\plate_3_H11_point_mutants_results.pkl",
        "rb") as f:
    plate_3_H11_results = pickle.load(f)
for peaks in plate_3_H11_results.selected_peaks.values():
    for peak in peaks:
        peak.plot_peak()
